# TP 1 — Estimer le prix des logements en Californie

**Régression — niveau débutant — environ 2 h — énoncé**

## Mise en situation

Tu viens d'être recruté par une agence immobilière qui couvre toute la Californie.
Aujourd'hui, quand un client demande une estimation, un expert se déplace : c'est lent et
cher. La direction te pose une question simple :

> **À partir des caractéristiques d'un quartier (revenu des habitants, âge des logements,
> taille des logements, position géographique), peut-on estimer automatiquement le prix
> médian d'un logement de ce quartier ? Et avec quelle marge d'erreur ?**




Les données sont publiques : ce sont celles du recensement américain de 1990, distribuées
avec scikit-learn (`fetch_california_housing`). Une ligne = un **quartier** (environ 1 400
habitants), et non un logement individuel.

## Ce que tu vas faire

| Étape | Ce que tu apprends |
|---|---|
| 1. Manipuler les données | charger un jeu de données, regarder sa taille, ses colonnes, ses types |
| 2. Explorer les données | comprendre la cible et repérer les variables qui comptent |
| 3. Préparer les données | séparer la cible (`y`) des variables explicatives (`X`) |
| 4. Nettoyer les données | valeurs manquantes, doublons, valeurs aberrantes |
| 5. Ton premier modèle | découper train / test, entraîner un modèle de référence et une régression linéaire |
| 6. Évaluer le modèle | MAE, RMSE, R², et l'analyse des erreurs |
| 7. Cross-validation | obtenir une estimation fiable, et non un score qui dépend de la chance |
| 8. Verdict final | évaluer une seule fois sur le jeu de test et répondre au client |

## Comment travailler

- Les cellules **À TOI DE JOUER** contiennent une consigne en commentaires : c'est à toi
  d'écrire le code en dessous.
- Respecte les noms de variables donnés dans les consignes : les exercices suivants les
  réutilisent. Si une variable manque, la cellule d'après ne tournera pas.
- Exécute les cellules **dans l'ordre**, de haut en bas.
- Chaque exercice se termine par un encadré **Ce que tu devrais observer** : c'est ton
  auto-correction. Si tu ne retrouves pas ces chiffres, relis ton code avant de continuer.
- Le corrigé complet est dans `01_tp_regression_prix_immobilier_corrige.ipynb`.

In [12]:
# Cellule a executer une fois, au debut du TP
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
print("Librairies chargées.")

Librairies chargées.


---
## Étape 1 — Manipuler les données

**Question de réflexion :** avant même de modéliser, quelles sont les trois premières
questions à se poser devant un fichier de données inconnu ?

Les trois réflexes : **combien de lignes et de colonnes ?**, **à quoi ressemble une
ligne ?**, **quel est le type de chaque colonne ?**

### Le dictionnaire des données

| Colonne | Signification |
|---|---|
| `MedInc` | revenu médian des ménages du quartier, en dizaines de milliers de dollars |
| `HouseAge` | âge médian des logements, en années |
| `AveRooms` | nombre moyen de pièces par logement |
| `AveBedrms` | nombre moyen de chambres par logement |
| `Population` | nombre d'habitants du quartier |
| `AveOccup` | nombre moyen d'occupants par logement |
| `Latitude` / `Longitude` | position géographique du quartier |
| `MedHouseVal` | **la cible** : prix médian d'un logement, en centaines de milliers de dollars |

Attention aux unités : `MedHouseVal = 2.5` signifie 250 000 dollars, et `MedInc = 3.87`
signifie 38 700 dollars de revenu médian.

### Exercice 1.1 — Charger et regarder

Charge le jeu de données, puis affiche ses dimensions, ses cinq premières lignes et le type
de chaque colonne.

Indices : `fetch_california_housing(as_frame=True).frame` renvoie un DataFrame pandas
complet (variables + cible). Ensuite : `.shape`, `.head()`, `.info()`.

In [13]:
from sklearn.datasets import fetch_california_housing

logements = fetch_california_housing(as_frame=True).frame

print("Dimensions (lignes, colonnes) :", logements.shape)
display(logements.head())
logements.info()


Dimensions (lignes, colonnes) : (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


**Ce que tu devrais observer** : 20 640 lignes et 9 colonnes (8 variables + la cible
`MedHouseVal`). Toutes les colonnes sont numériques (`float64`) et aucune n'est vide : c'est
un jeu de données confortable pour un premier TP. Une ligne décrit un quartier entier, donc
`AveRooms = 6.98` veut dire que les logements de ce quartier ont en moyenne 7 pièces.

### Exercice 1.2 — Interroger le tableau

Trois questions auxquelles pandas répond en une ligne chacune :

1. Quel est le prix médian le plus bas, le plus élevé, et la moyenne ? (`.describe()`)
2. Quels sont les 5 quartiers les plus chers ? (`.sort_values(..., ascending=False).head()`)
3. Combien de quartiers ont un revenu médian supérieur à 100 000 dollars, c'est-à-dire
   `MedInc > 10` ? (un filtre booléen, puis `len()`)

In [ ]:
# A TOI DE JOUER
# 1. Statistiques descriptives de toutes les colonnes
# 2. Les 5 quartiers les plus chers
# 3. Le nombre de quartiers ou MedInc depasse 10

**Ce que tu devrais observer** : le prix médian va de 0,15 (15 000 $) à 5,00 (500 000 $),

pour une moyenne de 2,07 (207 000 $). 

Il y a 308 quartiers dont le revenu médian dépasse
100 000 $.

Regarde bien les 5 quartiers les plus chers : ils valent tous **exactement 5.00001**. Ce
n'est pas un hasard, et c'est un point important pour la suite — garde-le en tête.

---
## Étape 2 — Explorer les données

**Question de réflexion :** avant d'entraîner quoi que ce soit, que faut-il regarder en
premier — les variables explicatives, ou la cible ?

**Toujours la cible d'abord.** C'est elle que le modèle devra reproduire : sa forme
détermine ce qui est possible.

### Exercice 2.1 — La distribution de la cible

Trace l'histogramme de `MedHouseVal` (50 barres), puis compte combien de quartiers ont un
prix supérieur ou égal à 5 et quelle part du total cela représente.

Indices : `logements["MedHouseVal"].hist(bins=50)`, puis un filtre booléen suivi de
`.sum()` et de `.mean()` (la moyenne d'un booléen est une proportion).

In [ ]:
# A TOI DE JOUER
# 1. Histogramme de la cible MedHouseVal avec 50 barres
# 2. Nombre et pourcentage de quartiers dont le prix est >= 5

**Ce que tu devrais observer** : la distribution est étalée vers la droite (quelques
quartiers très chers tirent la moyenne vers le haut), mais surtout une **barre anormalement
haute à 5,0** : 992 quartiers, soit 4,8 % du jeu de données.

Ce n'est pas la réalité : au moment du recensement, **tous les prix supérieurs à 500 000 $
ont été enregistrés comme 500 000 $**. On appelle cela une cible *censurée*, ou plafonnée.
Conséquence concrète : pour ces quartiers, la vraie valeur est inconnue, et aucun modèle ne
pourra les prédire correctement. On le verra noir sur blanc à l'étape 6.

### Exercice 2.2 — Quelles variables sont liées au prix ?

Calcule la corrélation de chaque variable avec `MedHouseVal`, triée de la plus forte à la
plus faible.

Indice : `logements.corr()["MedHouseVal"].sort_values(ascending=False)`.

Rappel : la corrélation vaut entre -1 et 1. Proche de 0, les deux variables n'évoluent pas
ensemble *de façon linéaire* — une relation en cloche passerait inaperçue.

In [ ]:
# A TOI DE JOUER
# Correlation de chaque colonne avec la cible, triee par ordre decroissant

**Ce que tu devrais observer** : `MedInc` domine largement (0,688). C'est logique côté
métier : **là où les gens gagnent plus, les logements valent plus**. Ensuite, tout
s'effondre : `AveRooms` 0,15, `HouseAge` 0,11, et le reste est proche de 0.

Attention au piège : `Latitude` (-0,14) et `Longitude` (-0,05) semblent inutiles. Pourtant
la position est un facteur clé en immobilier. La corrélation ne mesure qu'une relation
*linéaire* : ici, ce n'est pas « plus au nord = plus cher », c'est « au bord de l'océan =
plus cher ». L'exercice suivant le rend visible.

### Exercice 2.3 — Deux graphiques qui parlent

1. Un nuage de points `MedInc` (en x) contre `MedHouseVal` (en y). Sur 20 000 points le
   graphique serait illisible : travaille sur un échantillon de 2 000 lignes
   (`logements.sample(2000, random_state=42)`) avec `alpha=0.3`.
2. Une carte : `Longitude` en x, `Latitude` en y, la couleur donnée par le prix
   (`c=...`, `cmap="viridis"`, `s=5`). Tu devrais reconnaître la Californie.

In [ ]:
# A TOI DE JOUER
# 1. Nuage de points revenu / prix sur un echantillon de 2000 quartiers
# 2. Carte des quartiers coloree par le prix

**Ce que tu devrais observer** :

- Le nuage monte clairement de gauche à droite : plus le revenu est élevé, plus le prix
  l'est. Mais le nuage est **épais** : à revenu égal, les prix varient du simple au triple.
  Le revenu seul ne suffira donc pas.
- On voit aussi la **ligne horizontale à 5,0** : les quartiers plafonnés de l'exercice 2.1.
- La carte dessine la Californie, et les zones jaunes (chères) sont sur la côte, autour de
  San Francisco et de Los Angeles. **La géographie compte**, même si la corrélation ne le
  montrait pas.

---
## Étape 3 — Préparer les données

**Question de réflexion :** un modèle apprend une relation `y = f(X)`. Que mettre dans `X`,
et que mettre dans `y` ?

Convention universelle en machine learning :

- **`y`** : la cible, ce que l'on cherche à prédire. Ici `MedHouseVal`.
- **`X`** : les variables explicatives (*features*), tout ce dont on disposera **au moment
  de la prédiction**. Ici les 8 autres colonnes.

Ce dernier point est une règle de sécurité : si une colonne de `X` n'est connue qu'après
coup, le modèle trichera. Ici, les 8 variables décrivent le quartier et sont disponibles
avant toute vente : aucune ne triche. (Dans le TP de classification, tu verras un cas où
l'une d'elles triche, et les dégâts que cela provoque.)

### Exercice 3.1 — Séparer X et y

Construis `X` (toutes les colonnes sauf la cible) et `y` (la cible), puis vérifie leurs
dimensions.

Indice : `logements.drop(columns="MedHouseVal")` et `logements["MedHouseVal"]`.

In [ ]:
# A TOI DE JOUER
# Cree X (les 8 variables explicatives) et y (la cible), puis affiche leurs dimensions

**Ce que tu devrais observer** : `X` a 20 640 lignes et 8 colonnes, `y` a 20 640 valeurs.
`X` et `y` doivent **toujours** avoir le même nombre de lignes, dans le même ordre : la
ligne i de `X` décrit le quartier dont le prix est la valeur i de `y`.

---
## Étape 4 — Nettoyer les données

**Question de réflexion :** les données sont complètes et numériques. Y a-t-il quand même
quelque chose à nettoyer ?

Trois contrôles de routine : **valeurs manquantes**, **doublons**, **valeurs aberrantes**.
Les deux premiers sont mécaniques. Le troisième demande du bon sens métier : une valeur
aberrante est une valeur *impossible dans le monde réel*, pas simplement une grande valeur.

### Exercice 4.1 — Diagnostic

1. Combien de valeurs manquantes par colonne ? (`.isna().sum()`)
2. Combien de lignes dupliquées ? (`.duplicated().sum()`)
3. Affiche le minimum et le maximum de chaque colonne de `X`, et demande-toi lesquels sont
   physiquement impossibles pour un quartier.

In [ ]:
# A TOI DE JOUER
# 1. Valeurs manquantes par colonne
# 2. Nombre de lignes dupliquees
# 3. Min et max de chaque variable

**Ce que tu devrais observer** : aucune valeur manquante, aucun doublon. Mais trois maxima
sont absurdes :

- `AveRooms` = 141,9 pièces en moyenne par logement,
- `AveBedrms` = 34,1 chambres par logement,
- `AveOccup` = 1 243 occupants par logement.

Ce sont des quartiers minuscules ou atypiques (un hôtel, une caserne, quelques logements
seulement) où la moyenne n'a plus de sens. Quelques dizaines de lignes comme celles-là
suffisent à tirer une régression linéaire vers le décor.

### Exercice 4.2 — Retirer les lignes aberrantes

Construis un filtre qui garde les quartiers tels que `AveOccup < 10`, `AveRooms < 20` et
`AveBedrms < 5`, applique-le à `X` et à `y`, puis affiche le nombre de lignes retirées.

Indices : combine les conditions avec `&` (chacune entre parenthèses). Applique **le même
filtre** à `X` et à `y`, sinon les lignes ne se correspondent plus.

> Précision importante : on retire ici des lignes **manifestement fausses**, c'est une
> correction de qualité des données, et elle porte légitimement sur tout le jeu. En
> revanche, tout ce qui est un **choix de modèle** (remplacer les valeurs manquantes par une
> moyenne, encoder des catégories...) doit être calculé sur le seul jeu d'entraînement. Tu
> verras comment le faire proprement, avec un `Pipeline`, dans le TP de classification.

In [ ]:
# A TOI DE JOUER
# 1. Construis le masque valide : AveOccup < 10 ET AveRooms < 20 ET AveBedrms < 5
# 2. Applique-le a X et a y (resultats : X_propre et y_propre)
# 3. Affiche le nombre de lignes retirees et les nouvelles dimensions

**Ce que tu devrais observer** : 108 lignes retirées, il en reste 20 532. C'est 0,5 % du jeu
de données : on a éliminé le bruit sans sacrifier d'information.

Note qu'on **n'a pas touché aux quartiers plafonnés à 5,0**. Eux ne sont pas des erreurs :
ce sont de vrais quartiers chers, dont la valeur a été tronquée. Les supprimer reviendrait à
apprendre un modèle qui ignore les quartiers de luxe — ce que le client ne pardonnerait pas.

---
## Étape 5 — Ton premier modèle

**Question de réflexion :** si on entraîne un modèle sur toutes les données et qu'on mesure
ses erreurs sur ces mêmes données, que mesure-t-on exactement ?

Sa **mémoire**, pas sa capacité à généraliser. Un modèle qui a vu la réponse la récite. D'où
la règle d'or du machine learning :

> **On évalue toujours un modèle sur des données qu'il n'a jamais vues pendant son
> entraînement.**

On découpe donc en trois :

- **jeu d'entraînement** (64 %) : le modèle apprend dessus ;
- **jeu de validation** (16 %) : pour régler et comparer pendant le TP ;
- **jeu de test** (20 %) : **mis au coffre**, on n'y touchera qu'à l'étape 8, une seule fois.

Pourquoi un jeu de validation séparé du test ? Parce qu'à force de regarder un score, on
finit par choisir ce qui marche dessus. Le jeu de test doit rester une surprise, comme le
seront les vrais quartiers de demain.

### Exercice 5.1 — Découper les données

Utilise `train_test_split` deux fois : d'abord pour isoler 20 % de test, puis pour découper
ce qui reste en entraînement / validation (20 % de nouveau). Mets `random_state=42` à chaque
fois, pour que ton découpage soit reproductible.

In [ ]:
# A TOI DE JOUER
# 1. Importe train_test_split depuis sklearn.model_selection
# 2. X_train, X_test, y_train, y_test : 20 % pour le test, random_state=42
# 3. X_ap, X_val, y_ap, y_val : 20 % du train pour la validation, random_state=42
# 4. Affiche la taille des trois jeux

**Ce que tu devrais observer** : 13 140 quartiers pour apprendre, 3 285 pour valider,
4 107 au coffre. `random_state=42` fige le tirage aléatoire : avec la même valeur, tu
obtiendras toujours le même découpage, donc les mêmes scores que ce corrigé.

### Exercice 5.2 — Un modèle de référence, puis une régression linéaire

On entraîne **deux** modèles :

- `DummyRegressor(strategy="mean")` : il prédit toujours le prix moyen du jeu
  d'entraînement, sans rien apprendre. C'est le **niveau zéro** à battre. Un modèle qui ne
  fait pas mieux que lui ne sert à rien.
- `LinearRegression()` : la régression linéaire vue en cours. Elle cherche les coefficients
  qui minimisent l'erreur quadratique.

Entraîne les deux sur le jeu d'**apprentissage** avec `.fit(X_ap, y_ap)`, puis compare sur
les 5 premiers quartiers de validation les prix réels et les prix prédits.

In [ ]:
# A TOI DE JOUER
# 1. Importe DummyRegressor (sklearn.dummy) et LinearRegression (sklearn.linear_model)
# 2. Entraine naif et modele sur X_ap, y_ap
# 3. Construis un DataFrame comparant, pour les 5 premiers quartiers de validation :
#    le prix reel, la prediction du modele naif, celle de la regression lineaire

**Ce que tu devrais observer** : le modèle naïf annonce exactement la même valeur (2,07,
soit 207 000 $) pour les cinq quartiers — il ne regarde même pas `X`. La régression
linéaire, elle, propose une valeur différente pour chacun, parfois proche du prix réel,
parfois assez loin. Il faut maintenant mesurer cet écart sur les 3 285 quartiers de
validation, et non à l'œil sur cinq lignes.

---
## Étape 6 — Évaluer le modèle

**Question de réflexion :** comment résumer en un chiffre la qualité de 3 285 prédictions ?
Et ce chiffre répond-il à la question du client ?

Trois métriques de régression, à connaître :

| Métrique | Définition (en mots) | Se lit |
|---|---|---|
| **MAE** | moyenne des erreurs en valeur absolue | « on se trompe en moyenne de tant » — dans l'unité du client |
| **RMSE** | racine de la moyenne des erreurs au carré | comme la MAE, mais **pénalise lourdement les grosses erreurs** |
| **R²** | part de la variance de la cible expliquée par le modèle | 1 = parfait, 0 = aussi bon que prédire la moyenne, négatif = pire |

La MAE parle au client (« 50 000 $ d'erreur moyenne »). Le R² parle au data scientist. La
RMSE sert d'alarme : si elle est bien plus grande que la MAE, c'est que quelques prédictions
sont catastrophiques.

### Exercice 6.1 — Mesurer les deux modèles

Écris une fonction `evaluer(modele, X, y)` qui renvoie un dictionnaire avec la MAE, la RMSE
et le R², puis applique-la aux deux modèles sur le jeu de **validation**.

Indices : `mean_absolute_error`, `root_mean_squared_error` et `r2_score` sont dans
`sklearn.metrics`. Regroupe les deux résultats avec `pd.DataFrame([...], index=[...])`.

In [ ]:
# A TOI DE JOUER
# 1. Importe mean_absolute_error, root_mean_squared_error, r2_score
# 2. Ecris la fonction evaluer(un_modele, X_jeu, y_jeu) -> dict avec les cles MAE, RMSE, R2
# 3. Affiche un tableau comparant le modele naif et la regression lineaire sur la validation

**Ce que tu devrais observer** :

| | MAE | RMSE | R² |
|---|---|---|---|
| modèle naïf | 0,927 | 1,175 | -0,000 |
| régression linéaire | 0,489 | 0,667 | 0,678 |

La régression linéaire **divise l'erreur moyenne par deux** par rapport au modèle naïf :
48 900 $ d'erreur moyenne contre 92 700 $. Son R² de 0,678 signifie qu'elle explique 68 %
de la variabilité des prix ; le modèle naïf, lui, obtient un R² de 0, ce qui est la
définition même de « prédire la moyenne ».

Première réponse au client : **on sait estimer un quartier à environ 50 000 $ près**. Reste
à savoir si cette erreur est répartie uniformément, ou si elle se concentre quelque part.

### Exercice 6.2 — Où le modèle se trompe-t-il ?

Un score global cache toujours quelque chose. Calcule les **résidus** (erreur = prix réel −
prix prédit) sur la validation, puis :

1. trace leur histogramme (50 barres) ;
2. affiche la part des quartiers estimés à moins de 0,5 près (50 000 $) ;
3. compare l'erreur absolue moyenne **sur les quartiers plafonnés** (`y_val >= 5`) et sur
   les autres.

In [ ]:
# A TOI DE JOUER
# 1. residus = y_val - predictions du modele sur X_val
# 2. Histogramme des residus
# 3. Part des quartiers avec une erreur absolue < 0.5
# 4. Erreur absolue moyenne sur les quartiers plafonnes vs les autres

**Ce que tu devrais observer** :

- Les erreurs sont centrées sur 0 : le modèle ne surestime ni ne sous-estime
  systématiquement. C'est bon signe.
- **63 % des quartiers sont estimés à moins de 50 000 $ près.**
- Mais sur les 170 quartiers plafonnés, l'erreur moyenne est de **1,32** (132 000 $) contre
  **0,44** (44 000 $) ailleurs : **trois fois pire**. Le modèle sous-estime massivement les
  quartiers de luxe, exactement comme on l'avait anticipé à l'étape 2.
- À noter : la prédiction la plus élevée du modèle dépasse 5. Une régression linéaire ne
  connaît pas les bornes du monde réel ; elle extrapole.

**Ce que tu dirais au client** : « l'outil est fiable pour le marché courant, mais il faut
continuer à envoyer un expert pour les quartiers haut de gamme ». Ce genre de phrase vaut
plus qu'un R².

---
## Étape 7 — La cross-validation

**Question de réflexion :** le score de 0,678 vient d'**un seul** découpage
(`random_state=42`). Aurait-on obtenu le même avec un autre tirage ?

### Exercice 7.1 — Le score dépend-il de la chance ?

Écris une boucle sur `random_state` de 0 à 9 : à chaque tour, redécoupe le jeu
d'entraînement en apprentissage / validation, entraîne une `LinearRegression` et stocke le
R². Affiche ensuite le minimum, le maximum et l'écart entre les deux.

In [ ]:
# A TOI DE JOUER
# Boucle sur 10 graines aleatoires differentes, stocke les R2, affiche min et max

**Ce que tu devrais observer** : le **même** modèle, sur les **mêmes** données, obtient entre
0,646 et 0,666 de R² selon le seul hasard du découpage. Deux points de R² d'écart, gratuits.

Si tu compares deux modèles avec un seul découpage chacun, tu peux donc conclure à une
amélioration qui n'existe pas. C'est exactement le problème que résout la cross-validation.

### Exercice 7.2 — Cross-validation en 5 plis

Principe : on découpe le jeu d'entraînement en 5 morceaux (les *plis*). On entraîne 5 fois,
en laissant à chaque fois un pli différent de côté pour l'évaluation. On obtient 5 scores,
dont on regarde la **moyenne** (l'estimation) et l'**écart-type** (la stabilité).

Utilise `cross_validate` avec `KFold(n_splits=5, shuffle=True, random_state=42)` et les
métriques `["neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"]` sur `X_train` /
`y_train`.

Indice : scikit-learn maximise toujours ses scores, donc les erreurs y sont **négatives**
(d'où le préfixe `neg_`). Une MAE de -0,49 se lit « 0,49 d'erreur ».

In [ ]:
# A TOI DE JOUER
# 1. Importe cross_validate et KFold depuis sklearn.model_selection
# 2. Lance une cross-validation en 5 plis de LinearRegression sur X_train / y_train
# 3. Affiche, pour chaque metrique, la moyenne et l'ecart-type des 5 plis

**Ce que tu devrais observer** :

| Métrique | Moyenne | Écart-type |
|---|---|---|
| MAE | 0,493 | 0,009 |
| RMSE | 0,673 | 0,015 |
| R² | 0,660 | 0,012 |

L'estimation fiable de l'erreur moyenne est donc de **0,493, soit environ 49 000 $**, et
elle bouge peu d'un pli à l'autre (écart-type de 0,009) : le modèle est **stable**.

Deux habitudes à prendre dès maintenant :

1. On annonce toujours un score **avec son écart-type** : « 0,493 plus ou moins 0,009 ».
2. Une différence entre deux modèles n'a de sens que si elle **dépasse cet écart-type**.

---
## Étape 8 — Verdict final sur le jeu de test

**Question de réflexion :** le jeu de test est resté au coffre depuis l'étape 5. Pourquoi ne
pas s'en être servi plus tôt, et que va-t-il nous apprendre maintenant ?

Il sert à répondre à **une seule** question, **une seule fois** : le modèle tient-il sa
promesse sur des données totalement nouvelles ? Si on l'avait consulté à chaque essai, on
aurait fini par choisir ce qui lui plaît, et sa promesse ne vaudrait plus rien.

Dernière étape avant l'évaluation finale : on **ré-entraîne le modèle retenu sur tout le jeu
d'entraînement** (apprentissage + validation), pour qu'il bénéficie du maximum de données.

### Exercice 8.1 — L'épreuve de vérité

1. Entraîne une `LinearRegression` sur `X_train` / `y_train` en entier.
2. Évalue-la une seule fois sur `X_test` / `y_test` avec ta fonction `evaluer`.
3. Compare à l'estimation de la cross-validation et à celle du modèle naïf.
4. Affiche les coefficients du modèle pour interpréter ce qu'il a appris.

In [ ]:
# A TOI DE JOUER
# 1. modele_final = LinearRegression entrainee sur tout X_train / y_train
# 2. Evalue-la sur le jeu de test, et compare avec le modele naif
# 3. Affiche les coefficients : pd.Series(modele_final.coef_, index=X_train.columns)

**Ce que tu devrais observer** :

| | MAE | RMSE | R² | MAE en dollars |
|---|---|---|---|---|
| modèle naïf | 0,917 | 1,152 | -0,000 | 91 700 $ |
| régression linéaire | 0,494 | 0,673 | 0,658 | **49 400 $** |

Le point le plus important du TP : la cross-validation annonçait une MAE de **0,493**, le
jeu de test donne **0,494**. La promesse est tenue au millième près. C'est ce qui permet de
s'engager auprès d'un client **avant** la mise en production.

Les coefficients se lisent ainsi : `MedInc` vaut environ **+0,43**, donc 10 000 $ de revenu
médian supplémentaire dans un quartier correspondent à **+43 000 $ sur le prix médian** des
logements. `AveOccup` et les coordonnées géographiques ont des coefficients négatifs
importants : plus il y a de monde par logement, moins c'est cher ; et le prix baisse quand on
s'éloigne de la côte (latitude et longitude croissantes = vers le nord et vers l'est).

> Attention, deux pièges classiques avec les coefficients.
>
> 1. **Ils ne sont pas comparables entre eux** tant que les variables n'ont pas la même
>    échelle. `AveBedrms` affiche le plus gros coefficient (+0,79), mais il s'applique à un
>    nombre de chambres qui varie de 1 à 2, alors que `MedInc` varie de 0,5 à 15. Pour
>    comparer les influences, il faudrait d'abord standardiser les variables.
> 2. **Un coefficient n'est pas une cause.**
>    Augmenter artificiellement `AveRooms` ne ferait pas monter les prix : un modèle constate
>    des associations, il ne démontre pas de mécanisme.

---
## La réponse au client

> **Oui, on peut estimer automatiquement le prix médian d'un quartier californien, avec une
> erreur moyenne d'environ 50 000 $** (contre 92 000 $ si on annonçait bêtement le prix
> moyen du marché). Le modèle explique environ 66 % de la variation des prix, il est stable
> d'un échantillon à l'autre, et son principal facteur est le revenu des habitants.
>
> **Deux limites à annoncer honnêtement** : il sous-estime fortement les quartiers de luxe
> (les données sources y sont plafonnées à 500 000 $), et il repose sur un recensement de
> 1990 — il faudrait le ré-entraîner sur des données actuelles avant tout usage réel.

## À retenir

1. **La cible d'abord.** Son histogramme a révélé le plafond à 500 000 $, qui explique la
   principale faiblesse du modèle. Cinq minutes d'exploration ont économisé des heures de
   doute.
2. **Un modèle se juge face à une référence.** Sans le `DummyRegressor`, un R² de 0,66 ne
   veut rien dire.
3. **Choisir la métrique qui parle au métier.** La MAE en dollars a convaincu le client, pas
   le R².
4. **Un score unique est un score fragile.** Le même modèle valait entre 0,646 et 0,666 de
   R² selon le découpage : la cross-validation donne une estimation, et sa stabilité.
5. **Le jeu de test ne se consulte qu'une fois.** C'est ce qui rend la promesse crédible.

## Pour aller plus loin

Ces pistes n'ont pas de corrigé : c'est à toi de jouer.

1. **Ajoute une variable** : crée `pieces_par_personne = AveRooms / AveOccup` et regarde si
   la MAE en cross-validation baisse de plus d'un écart-type.
2. **Change de modèle** : remplace `LinearRegression` par
   `DecisionTreeRegressor(max_depth=8)` ou `RandomForestRegressor(n_estimators=100)`.
   Compare en cross-validation, sur le jeu d'entraînement uniquement.
3. **Traite le plafond** : entraîne le modèle en excluant les quartiers à 5,0 et regarde
   l'effet sur les autres quartiers. Est-ce honnête vis-à-vis du client ?
4. **Regarde les pires erreurs** : affiche les 10 quartiers du jeu de test les plus mal
   estimés. Ont-ils un point commun ?